In [1]:
import sys
from pathlib import Path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# Auto-generated by scripts/build.py — do not hand-edit.
# Rebuild with: python scripts/build.py worksheets/lesson_10_1_betweenness_girvan_newman.yaml

from engine import Worksheet
from engine.plot_style import apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_10_1_betweenness_girvan_newman.yaml')
ws.start()


# Community Discovery I: Betweenness and the Girvan–Newman Method

_Session c57eade9_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

_Continuing your earlier session — **20** answer(s) restored from this workspace._

In [2]:
ws.render_section('intro')

## Finding the groups nobody labeled

Let's start with three ideas from earlier in the course, since all three
point the same way. **Homophily** says we tend to be similar to our
friends. **Triadic closure** says two people with a friend in common are
more likely to become friends. **Weak ties** (acquaintances, in the sense
of Chapter 3) are the links that tend to reach *outside* your immediate
circle, because a link that bridges two close-knit circles is rarely a
close friendship.

Now put the three together, and we get a prediction about the shape of a
whole network: dense pockets of people who all know each other, joined to
one another by a much smaller number of links. Consider the friendships
among the students in a large dorm. We would expect a pocket for each floor or
each team, and only a few friendships running between the pockets. In this
worksheet we turn that picture into an algorithm, one that takes a network
nobody has labeled and hands back the pockets.

### The strategy: find the links *between* groups, and cut them

Let's look for the edges that join the groups, instead of for the groups
themselves. When a single edge joins two pockets, it carries every journey
between them; when several do, they share that traffic. An edge inside a
pocket, in contrast, has plenty of routes around it. Thus we measure, for
each edge, how much traffic it carries (its *betweenness*), and we delete
the busiest edges until the network falls apart into pieces; the pieces
are the communities. Girvan and Newman (2002) proposed this method, and
Easley and Kleinberg teach it in Section 3.6 as the **Girvan–Newman
method**.

### Notation, stated once and used throughout

- $n$ is the number of nodes, and $m$ is the number of edges.
- $k_i$ is the **degree** of node $i$: how many edges touch it.
- $\sigma(v)$ is the number of **different shortest paths** from whichever
  node we are currently starting from (the *root*) to node $v$.
- The **flow** on an edge is the share of one root's traffic that the edge
  carries, where the root sends one unit to each other node it can reach
  (a node in another piece of the network gets nothing).
- The **betweenness** of an edge is those flows added up over all the
  roots, then halved.
- $Q$ is **modularity**, a score for how good a proposed division into
  communities is; we define it in the second half of this worksheet.
  $Q$ runs from $-\frac{1}{2}$ to just under $1$. A score near $0$ means
  the division is no better than a random baseline that keeps each
  node's degree, a negative score means worse, and the Lesson 10 notes
  put networks with strong community structure at about $0.3$ to $0.7$.

In the second worksheet we take $Q$ apart and maximize it directly, with
linear algebra instead of cutting.

Source: Michelle Girvan and M. E. J. Newman (2002). Community structure in
social and biological networks. *Proceedings of the National Academy of
Sciences* 99(12): 7821–7826.
DOI: [10.1073/pnas.122653799](https://doi.org/10.1073/pnas.122653799)


In [3]:
ws.render_section('concept_why_communities')

### Why we expect communities, and what makes them findable

Let's start from the picture the three ideas predict: two dense pockets
joined by a thin waist. Girvan and Newman (2002) built their method on one
observation about this picture, that the few edges between the pockets carry
the traffic between them. Why should a few edges carry so much? Every trip
from one pocket to the other has to cross the waist somewhere.

In the figure we color the nodes by group, label each edge that runs between
the pockets with the shortest-path traffic it carries, and score the
two-group division with $Q$ (defined later on this sheet). Drag the control
to add routes between the pockets and watch both numbers. Notice that the
groups stay as dense as they were; what changes is how easy they are to tell
apart.


A community boundary is easiest to see when there are few ways across it.
With one crossing, that edge carries each trip between the two sides, and
its number stands far above the rest. Add crossings and, on this network,
the trips divide among them; no single edge stands out, and the two-group
split scores lower as well.

Does the number on a crossing always fall when we add another one? On
this network both signals fall, but on other networks the betweenness
need not. Consider a left group of three nodes joined in a path,
$u$–$w$–$v$, and a right group $x$–$z$–$y$, with crossings $v$–$x$ and
$w$–$x$. Adding a third crossing, $u$–$x$, raises the number on $v$–$x$
from $3$ to $7/2$, because the pair $u$ and $v$ gains a second shortest
path, $u$–$x$–$v$, that runs out through the new crossing and back
through $v$–$x$.


In [4]:
ws.render_section('q_1')

**Q1 · easy**

What structural shape are community-detection methods looking for in a
network?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** We define a community by where the edges are, dense inside and sparse
between. A group's size and its members' degrees don't enter. For example,
equal degrees describe a regular network, which can have no community
structure at all, and real communities come in very different sizes. If the
groups have no path between them, they are separate components, and we need
no algorithm to find them. The hard and interesting case is the one where
the groups are connected, just not densely.


In [5]:
ws.render_section('q_2')

**Q2 · medium**

Three of these statements correctly connect earlier ideas from the course
to the expectation that networks contain communities. **One is not.**
Which statement is incorrect?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** The statement about deleting weak ties has the consequence backwards. A weak
tie may carry little conversation, but a link that reaches outside your
pocket can carry a great deal of shortest-path traffic. If it is one of only
a few links between two pockets, all the journeys between them have to share
those few links. Thus deleting the weak ties can split a network into
pieces, and the algorithm in this worksheet hunts for such links first. As
for the other three statements, they are the standard picture. Similarity
builds pockets, closure thickens them, and the links that reach outward,
which tend to be weak ties (Easley and Kleinberg, Section 3.2), join them
up.


In [6]:
ws.render_section('q_3')

**Q3 · easy**

The **betweenness of an edge** measures which of the following?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Betweenness measures traffic, and shortest paths define the traffic. That
is, each pair of nodes exchanges one unit, split evenly when several
shortest paths tie, and an edge's betweenness is the total it ends up
carrying. If we count the edges at its two ends instead, we get a
degree-like number that says nothing about routes. A longest path through an
edge can be very long in a large network, and its length has nothing to do
with how heavily the edge is used. Counting triangles measures nearly the
opposite property. An edge in many triangles sits deep inside a dense
pocket, and that is the kind of edge betweenness scores low.


In [7]:
ws.render_section('q_4')

**Q4 · medium** · 💡

In the network shown, which edge carries the most shortest-path traffic,
and why?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Let's try removing the edge between R and S. The network falls into two
halves, and thus R–S is the only route between them; each left-to-right
journey has to use it. In contrast, the edge joining P and Q is the least
used. P and Q are neighbors of each other and of R, and the only pair
whose shortest path needs P–Q is P and Q themselves. Degree is not the
criterion either, since an edge at a high-degree node inside a dense
pocket has routes around it. Symmetry does not make the edges equal
either. The network is symmetric left to right, which makes matching
edges on the two sides equal to each other, not equal to the bridge
between them.


In [8]:
ws.render_section('q_5')

**Q5 · medium** · 💡

In the network shown, exactly one edge separates the three left-hand nodes
from the three right-hand nodes. What is that edge's betweenness, the total
shortest-path traffic it carries when each pair of nodes exchanges one unit?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Let's count the pairs that use that edge. Each of the three nodes on the
left needs to reach each of the three on the right, and each of those
journeys has a single route, through that edge. That is, there are $3 \times
3 = 9$ such pairs, each contributing one unit, and the betweenness is $9$.
If you picked $6$, you added the two sides instead of multiplying them,
which counts nodes instead of pairs. The value $15$ counts every pair of
nodes in the six-node network, $\frac{6 \times 5}{2}$, but most of those
pairs sit on one side and never cross. If you counted only the three nodes
on one side, you forgot that each of them has three destinations.


In [9]:
ws.render_section('concept_bfs_counts')

### Steps 1 and 2: layers, then counting shortest paths without listing them

How can we find the betweenness of all the edges without listing all the
shortest paths in the network? Let's follow the eight steps of the Lesson
10 notes on the network Easley and Kleinberg work through in Section 3.6
(their Figure 3.18): Steps 1 and 2 here, Steps 3 to 6 (the upward pass)
in the next cell, and Steps 7 and 8 after that.

We pick node A as the root and run a breadth-first search from it,
drawing the layers downward as the book does. Then, beside each node, we
write $\sigma(v)$, the number of different shortest paths from A to that
node. Step the control to reveal one layer at a time, and watch how each
count is built: never by listing paths, always by adding up the counts of
the nodes directly above.

Dashed edges join two nodes in the same layer. Such an edge lies on no
shortest path from A, and thus it adds nothing to A's counts or flows.
From other roots such an edge does carry traffic.


As we have seen, the number of shortest paths to a node obeys a rule that
looks only one layer up. Thus a question about whole paths, which can be
very many, becomes one addition per node, done in a single downward sweep.
We never list the six shortest paths to K; we count them as $3 + 3$, and
those 3s were counted the same way.

However, the rule rests on each edge being one step long. Consider a road
network whose edges have different lengths (travel times, say). There,
layers by step count no longer sort the nodes by distance, and counting
shortest paths needs a different search.


In [10]:
ws.render_section('q_6')

**Q6 · easy**

The efficient way to compute betweenness starts by running a breadth-first
search from a chosen starting node, called the **root**. What does
breadth-first search give us that we need?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Since a node in layer $d$ is $d$ steps from the root, and any shortest path
to it descends one layer per step, the layering is what the next steps need;
they rely on that fact to look only one layer up. Breadth-first search does
not discard equally short paths, and it must not, because betweenness splits
the traffic evenly among them. Degree order plays no part in the search.
Breadth-first search deletes no edges either. An edge joining two nodes in
the same layer stays in the network, since it lies on no shortest path from
this root but can lie on shortest paths from others.


In [11]:
ws.render_section('q_7')

**Q7 · medium** · 💡

The figure shows a breadth-first search from the highlighted root $r$,
with the layers running downward. Using the rule that a node's count of
shortest paths is the **sum of the counts of the nodes directly above
it**, how many different shortest paths run from $r$ to node $e$?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Let's work downward. Nodes $a$ and $b$ each sit next to the root, and
each is reached by one shortest path. Node $c$ has both $a$ and $b$
above it and collects $1 + 1 = 2$; node $d$ has only $b$ above it and
collects $1$. Then node $e$ has $c$ and $d$ above it, which gives $2 + 1
= 3$. If you picked $2$, you counted the nodes above $e$ instead of the
paths that run through them. Picking $4$ adds an extra $1$ for $e$
itself, but the rule counts paths arriving, and a node adds nothing to
its own count. If we kept one incoming count and dropped the other, we would
get $1$, as if only one route could be used.


In [12]:
ws.render_section('q_8')

**Q8 · easy** · 💡

In the same breadth-first search from root $r$, how many different
shortest paths run from $r$ to node $d$, and why?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Node $d$ has a single node above it, $b$, and $b$ is a neighbor of the root
with a count of one. Thus $d$ inherits a count of one. Does being in the
second layer decide the count? Consider $c$, the other node in that layer,
which has two nodes above it and a count of two; that contrast is what the
rule captures. How many neighbors $b$ has plays no part, since the rule adds
the counts of the nodes above, not their edges. Also, $d$ does lie on
shortest paths to the lower nodes. Even if it didn't, its own count would
not be zero, because each node the root can reach has at least one shortest
path to it.


In [13]:
ws.render_section('q_9')

**Q9 · easy**

Three of these statements about the layers of a breadth-first search are
correct. **One is not.** Which statement is incorrect?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Which root we start from decides which edges are used, and the betweenness
of an edge is a sum over all the roots. An edge whose ends land in the same
layer from one root sits on shortest paths from other roots. Thus deleting
it changes betweenness values, and it can change distances in the network
too. However, the other three statements describe the layering property that
makes the method work. Indeed, we draw a same-layer edge dashed while the
search from that root is under way because it lies on no shortest path from
the root.


In [14]:
ws.render_section('concept_flow_up')

### Steps 3 to 6: pushing the traffic back up, split by the path counts

Now we route the traffic. We let the root send one unit to each other node
it can reach, and we work out how that traffic spreads over the edges, this
time climbing upward from the deepest layer (Steps 3 to 6 of the Lesson 10
notes).

Let's follow a single node as the flow climbs. Each node's total is one unit
for itself plus everything arriving from below, and the node divides that
total over the edges leading upward in proportion to the shortest-path
counts $\sigma$ from Step 2. That is, a node above that is reached by twice
as many shortest paths receives twice as much. Why should the split follow
the counts? Each shortest path from the root to the node carries an equal
share of its traffic, and a node above with twice the count has twice as
many of those paths running through it.

This figure shows the flows on the edges; the counts $\sigma$ are in the
figure of the cell above, and each caption lists them. Step the control to
climb one layer at a time.


Notice that the two sweeps are mirror images, and they need each other.
While the downward sweep counts how many ways the traffic could have come,
the upward sweep uses those counts as the proportions for sending it back.
Thus one root costs a single pass down and a single pass up, and that is
what makes betweenness affordable.

However, the check at the root, $n - 1$ units in all, holds for a connected
network only. If the network is cut into pieces, the edges at the root carry
one unit for each node the root can reach, as we will see in the cascade
below once the network has been cut.


In [15]:
ws.render_section('q_10')

**Q10 · easy**

When we compute the flows from one root, how much traffic is sent, and to
where?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** We send one unit per destination. Thus a root in a connected network of
$n$ nodes sends $n - 1$ units in all, and the flows on the edges
leaving the root add up to $n - 1$, the arithmetic check on the whole
pass. If the root sent one unit in total, each root's contribution would
be the same size whatever the network's size, and we would lose that check.
A unit to each neighbor is a purely local quantity that just recovers
degree. If we sent one unit down each edge at the root, we would ignore the
destinations, and the destinations are what the proportional splitting
further down is about.


In [16]:
ws.render_section('q_11')

**Q11 · hard** · 💡

In the search from root $r$, node $e$ has two nodes above it: $c$, which is
reached by 2 shortest paths from $r$, and $d$, which is reached by 1. Node
$e$'s total flow (one unit for itself plus everything arriving from below)
is $2$. How does that $2$ divide over the two edges leading up from $e$?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Since the counts are 2 for $c$ against 1 for $d$, the total of $2$
splits in the ratio $2 : 1$, and twice as much goes the way that twice
as many shortest paths came. That is, $\frac{2}{3} \times 2 =
\frac{4}{3}$ goes up to $c$ and $\frac{1}{3} \times 2 = \frac{2}{3}$ up
to $d$, and the two add back to $2$. An even split ignores the counts,
and it would be right only if the two upper nodes were reached equally
often. Sending the larger share to $d$ inverts the ratio. If we put the
full $2$ on both edges, we would duplicate the traffic instead of dividing
it, and the flow would grow each time it climbed a layer.


In [17]:
ws.render_section('q_12')

**Q12 · medium** · 💡

Working upward, we reach a node $X$ that already has $\frac{5}{2}$ units of
flow arriving on the edges below it. What total does $X$ divide over the
edges leading upward from it?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Traffic is added at each node, not conserved through it, since the root
sends a unit to $X$ as well as to each node beyond $X$. Where does
$X$'s own unit travel? On its way from the root it rides the edges above
$X$, and it travels on no edge below $X$, since it stops there. Hence
$\frac{5}{2} + 1 = \frac{7}{2}$. If we treated flow as conserved, we would
forget $X$'s own unit, and the edges at the root would fall short of the
$n - 1$ units the root sends. Subtracting a unit has the sign backwards.
A fixed $1$ per node discards the pass-through traffic, and the
pass-through traffic is how deeper parts of the network affect an edge
near the root.


In [18]:
ws.render_section('q_13')

**Q13 · medium** · 💡

In a **connected** network of $n$ nodes, we run the flow computation from
one root. What do the flows on the edges touching that root add up to, and
why?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Consider the units the root sends. Each one has to leave along one of the
edges that touch the root, and there is one unit per destination. Thus the
total is the number of destinations, $n - 1$. We run this check each time,
since it catches an arithmetic slip anywhere in the upward pass. Using $n$
counts the root as a destination for itself, and it is not one. A root's
degree counts the edges available instead of the traffic on them, and those
edges carry very different amounts. As for $m$, the edge count is a property
of the whole network with no reason to match one root's traffic.


In [19]:
ws.render_section('concept_sum_over_roots')

### Steps 7 and 8: every node takes a turn, then halve

One root gives us one node's view of the network, but betweenness adds
up the traffic between all pairs of nodes. Let's repeat the flow
computation with each node in turn as the root and add the results on
each edge. Step the control through the roots to watch the totals build
up, then take the last setting to see the halving. Orange nodes are the
roots used so far, the orange edge is the current leader, and an edge
with no number has collected nothing yet. The last setting shows the
halved totals, and it marks no roots.


Notice that the halving corrects a double count. In an undirected network
the trip from $X$ to $Y$ and the trip from $Y$ to $X$ are the same trip, but
we count it twice when we sum over the roots, once from each end. An edge's
importance can't depend on which end of a trip we call the start, and thus
we collapse the two counts into one. In a directed network the two trips are
different trips, and there we would not halve.


In [20]:
ws.render_section('q_14')

**Q14 · medium** · 💡

After running the flow computation with every node in turn as the root and
adding the results on each edge, the final step divides every total by 2.
What is being corrected?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Let's follow one pair, $X$ and $Y$. Since the sum runs over roots, the
pair contributes once when $X$ is the root and again when $Y$ is, and in
an undirected network the shortest paths between them are the same set
both times. An edge's importance should not depend on which end of a trip
we call the start, and thus the two contributions are one fact recorded
twice. An edge's two ends have nothing to do with it, since we collect
the flow per edge, from each root's traffic. An undirected network has
no backwards shortest paths. Each unit the root sends is delivered; the
root is not one of its own destinations, and the count $n - 1$ already
accounts for that.


In [21]:
ws.render_section('q_15')

**Q15 · medium** · 💡

The figure shows the finished betweenness value on each edge. The two
edges labeled $1$ are the ones joining the two degree-2 nodes on each
side. Why is $1$ the smallest value any edge in a connected network can
have?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Any two adjacent nodes are one step apart, and one step is as short as a
path can be. Thus the edge between them is the unique shortest path for that
pair and collects its full unit. Everything else an edge carries comes on
top of that floor. However, the claim that each edge lies on the shortest
paths of exactly one pair gets the right number for the wrong reason, since
most edges serve many pairs; look at the value $9$ on the bridge in this
same figure. Betweenness is not rounded at all. For example, it takes
fractional values whenever shortest paths tie, as several values in this
lesson do. An edge in a triangle has no fixed betweenness either: P–R sits
in a triangle and carries $4$.


In [22]:
ws.render_section('q_16')

**Q16 · medium** · 💡

The network shown is the one from the concept cells with a **second**
crossing added, joining $C$ and $G$ (both crossings are highlighted).
Before the second crossing existed, the edge joining $B$ and $D$ had
betweenness $12$. What has happened to that edge's betweenness now, and
why?


Output()

_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Consider the twelve left-to-right pairs. With one crossing, all of them were
forced onto that single edge, and a second crossing gives many of those
pairs a route that is as short or shorter. Thus the traffic divides, and the
original edge's share drops sharply (to $41/6$). Deleting an edge moves
traffic around in the same way, and we recompute each betweenness after each
deletion for this reason, instead of reusing the first pass. Counting pairs
is not enough, since the value depends on the routes between them, and the
routes are what changed. In a connected network, adding an edge always
lowers the average betweenness per edge, since the total over all edges is
the sum of all the pairwise distances, which cannot grow, while the number
of edges grows by one; a single edge's number can still rise. Also, the
split is not even, since the two crossings sit at different places in the
network and serve different pairs, and no symmetry forces them to be equal.


In [23]:
ws.render_section('concept_gn_cascade')

### The Girvan–Newman cascade, and where to stop it

Now let's run the method itself. We find the busiest edge, delete it,
recompute each betweenness from scratch, and repeat; the communities at any
moment are the connected pieces that remain. Once the network is in pieces,
a root sends traffic only to the nodes in its own piece (for example, after
the first cut A's traffic reaches only B and C). While there are two to four
pieces, the node colors mark them, and a removed edge stays in the figure in
light gray.

Step the control to watch the network come apart. How much do the numbers on
the surviving edges move after each deletion, and what does the score $Q$
(defined later on this sheet) do along the way? Notice that the method never stops on its own, and we will have to decide when to stop it.


A deletion can only break a piece apart; it never joins two. Thus the
divisions the cascade produces sit inside one another, and each stage
refines the one before it. We are left with a whole family of answers, and
to choose among them we need something the cascade does not contain. Nesting
has a second cost, since at each stage a node lies in one piece only, a
limit we take up in the clique cells at the end of this sheet.


In [24]:
ws.render_section('q_17')

**Q17 · medium**

Three of these statements about the Girvan–Newman method are correct.
**One is not.** Which statement is incorrect?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Deleting an edge reroutes each journey that used it. Thus the
betweenness of the surviving edges can change sharply, and an edge
ranked low can jump to the top, because it now carries traffic that used
to go another way. We recompute after each deletion for this reason, and
recomputing is what makes the method expensive. Each of the other three
statements holds. Recomputing all the values is Easley and
Kleinberg's rule (p. 68); ties are broken by removing all the tied
edges at once instead of picking one; and since deletions only ever
break pieces apart, the sequence of divisions is nested.


In [25]:
ws.render_section('q_18')

**Q18 · easy**

Run the Girvan–Newman method to completion, deleting edges until none are
left. What does it produce?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Each deletion either breaks a piece in two or leaves the pieces as they
were. Thus running the method to the end walks the network from one piece
down to $n$ pieces, a whole hierarchy of divisions instead of one answer.
Which of them should we keep? We need something outside the method to answer
that, and modularity fills the gap. If we stopped after the first split, we
would throw away each finer division. Also, the output is not a ranking of
the original edges. Since betweenness is recomputed after each deletion, the
order in which edges are removed can differ a great deal from their ranking
at the start.


In [26]:
ws.render_section('q_19')

**Q19 · medium** · 💡

Why is the Girvan–Newman method expensive to run on a large network?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Consider the work in one betweenness computation, a sweep down and up from
each node as root. We need a fresh one after each of the method's many
deletions, and that product of the two costs limits the method to networks
of moderate size. Listing shortest paths explicitly is what the
layer-counting trick avoids, and avoiding it is the point of the second half
of Section 3.6. Trying each possible division is a different and far worse
approach that this method never takes. As for breadth-first search, it is
the fast option here; what hurts is the number of times it runs.


In [27]:
ws.render_section('q_20')

**Q20 · medium** · 💡

Since the Girvan–Newman method keeps cutting until nothing is left,
something outside the method has to decide which of its divisions to keep.
What is the sensible criterion?


_Restored from your earlier session — already answered. Your recorded credit is shown below; you don't need to redo this problem._

**✓ Correct (1 try)** — recorded credit 1.00

**Rationale:** Since the cascade produces a whole family of divisions, choosing among them
needs a score that we can compute for any division and compare across them,
and modularity is one. The first cut is often the right answer, but nothing
guarantees it; a network with three genuine groups needs at least two
splits, and only a score can tell you so. Equal group sizes are not a
property real communities have. If we kept the last division before the
network becomes disconnected, we would keep the whole network in one piece,
the division the method started from.


In [28]:
ws.render_section('concept_expected_edges')

### The null model: how many edges *should* join two nodes?

Before we can score a division, we need to say what "more connected than you
would expect" means. But expected under what model of the network? We need a
baseline, and we build it by destroying the network and rebuilding it at
random.

Let's cut each edge in half. Node $i$ is left holding $k_i$ loose
half-edges, called **stubs**, and the whole network has $2m$ of them, since
each of the $m$ edges contributed two. Now we pair the stubs up blindly,
each stub with one of the others chosen at random. Worksheet 8.2 built
random networks the same way, with its pairing construction; used as a
baseline, this random rewiring is called the **configuration model**. We use it as a null model, which makes no prediction; it shows what a network with these same degrees and no community structure
would do on average.

Move the controls and watch how many $i$–$j$ edges the rewiring produces on
average, and compare the exact chance that a stub lands on $j$ with the
simpler one modularity uses.


As we built it, the baseline keeps each node's degree and throws away
everything else. Why keep the degrees fixed? A hub has many edges in the
real network and many in the baseline, and thus it gets no credit merely for
being a hub. What is left after the subtraction is structure that degree
alone cannot explain, and a community is one kind of it.

However, the baseline's $k_i k_j / 2m$ is the large-network form. On a small
network it sits a little below the exact $k_i k_j / (2m - 1)$ (for example,
$2/3$ against $12/17$ at the opening setting), and for a pair of hubs it can
exceed 1, a count of repeated edges that no simple network allows (blind
pairing can join two nodes more than once).


In [29]:
ws.render_section('q_21')

**Q21 · easy**

In the stub picture, a network with $m$ edges is cut apart into loose
half-edges. How many half-edges are there in total, and why?


Output()

In [ ]:
ws.render_section('q_22')

In [ ]:
ws.render_section('q_23')

In [ ]:
ws.render_section('q_24')

In [ ]:
ws.render_section('concept_modularity_score')

In [ ]:
ws.render_section('q_25')

In [ ]:
ws.render_section('q_26')

In [ ]:
ws.render_section('q_27')

In [ ]:
ws.render_section('q_28')

In [ ]:
ws.render_section('q_29')

In [ ]:
ws.render_section('q_30')

In [ ]:
ws.render_section('concept_clique_communities')

In [ ]:
ws.render_section('q_31')

In [ ]:
ws.render_section('q_32')

In [ ]:
ws.render_section('q_33')

In [ ]:
ws.render_section('q_34')

In [ ]:
ws.render_section('concept_resolution_knob')

In [ ]:
ws.render_section('q_35')

In [ ]:
ws.render_section('q_36')

In [ ]:
ws.render_section('q_37')

In [ ]:
ws.render_section('q_38')

In [ ]:
ws.render_section('concept_partition_vs_cover')

In [ ]:
ws.render_section('q_39')

In [ ]:
ws.render_section('q_40')

In [ ]:
ws.render_section('q_41')

In [ ]:
ws.render_section('q_42')

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()